<a href="https://colab.research.google.com/github/smosharof/Resume.Walkthrough/blob/main/Personalized_Recommendation_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#Generate Synthetic Code
import pandas as pd
import numpy as np

# --- Parameters (Adjust as needed) ---
num_users = 1000
num_items = 200
num_purchases = 5000
categories = ['Tops', 'Bottoms', 'Shoes', 'Accessories', 'Outerwear']
brands = ['BrandA', 'BrandB', 'BrandC', 'BrandD']
colors = ['Red', 'Blue', 'Green', 'Black', 'White']
sizes = ['S', 'M', 'L', 'XL']

# --- User Data ---
users = pd.DataFrame({
    'user_id': range(1, num_users + 1),
    'age': np.random.randint(16, 70, num_users),
    'gender': np.random.choice(['Male', 'Female'], num_users),
    'location': np.random.choice(['Urban', 'Suburban', 'Rural'], num_users)
})

# --- Item Data ---
items = pd.DataFrame({
    'item_id': range(1, num_items + 1),
    'category': np.random.choice(categories, num_items),
    'brand': np.random.choice(brands, num_items),
    'color': np.random.choice(colors, num_items),
    'size': np.random.choice(sizes, num_items),
    'price': np.random.uniform(10, 200, num_items)
})

# --- Interaction Data (Purchases) ---
interactions = pd.DataFrame({
    'user_id': np.random.choice(users['user_id'], num_purchases),
    'item_id': np.random.choice(items['item_id'], num_purchases),
    'purchase_date': pd.to_datetime(np.random.choice(pd.date_range(start='2023-01-01', end='2024-12-31', freq='D'), size=num_purchases)), # Adjust the length of 'purchase_date'
    'rating': np.random.choice([4, 5], num_purchases),  # Simulate positive purchases
    'quantity': np.random.randint(1, 3, num_purchases)
})
interactions['purchase_date'] = interactions['purchase_date'].sample(frac=1).reset_index(drop=True) # Shuffle dates

# --- Merge Data for Combined Analysis ---
data = pd.merge(interactions, users, on='user_id')
data = pd.merge(data, items, on='item_id')

print("Users Data:")
print(users.head())
print("\nItems Data:")
print(items.head())
print("\nInteractions Data:")
print(interactions.head())
print("\nMerged Data:")
print(data.head())
print(data.info())

Users Data:
   user_id  age  gender  location
0        1   64    Male     Rural
1        2   21  Female     Urban
2        3   61    Male     Urban
3        4   25  Female  Suburban
4        5   17    Male     Rural

Items Data:
   item_id   category   brand  color size       price
0        1    Bottoms  BrandD  Green    S   69.120158
1        2  Outerwear  BrandD  Black    S  150.909247
2        3  Outerwear  BrandB  Black    L  181.305030
3        4    Bottoms  BrandC    Red    M  160.105963
4        5      Shoes  BrandA  Black    M  177.017847

Interactions Data:
   user_id  item_id purchase_date  rating  quantity
0      747       26    2024-11-23       4         2
1      650        6    2024-07-25       4         1
2      763      131    2023-06-12       5         2
3      882       15    2024-08-08       5         1
4      953      123    2024-08-30       5         1

Merged Data:
   user_id  item_id purchase_date  rating  quantity  age  gender  location  \
0      747       26    

In [3]:
# Collaborative and Content based filtering
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix

# --- Collaborative Filtering (User-Based) ---
def get_user_item_matrix(df):
    """Creates a user-item matrix from the interaction data."""
    user_item_matrix = df.pivot_table(index='user_id', columns='item_id', values='rating', fill_value=0)
    return user_item_matrix

def get_user_similarity(user_item_matrix):
    """Calculates user similarity using cosine similarity."""
    user_similarity = cosine_similarity(user_item_matrix)
    return user_similarity

def get_user_based_recommendations(user_id, user_item_matrix, user_similarity, items_df, top_n=10):
    """Generates user-based collaborative filtering recommendations."""

    if user_id not in user_item_matrix.index:
        print(f"User {user_id} not found in user-item matrix. Returning popular items.")
        return items_df.sort_values(by='price', ascending=False).head(top_n)  # Simple fallback

    similar_users = pd.Series(user_similarity[user_item_matrix.index.get_loc(user_id)], index=user_item_matrix.index)
    similar_users = similar_users.sort_values(ascending=False)
    similar_users = similar_users.drop(user_id)  # Remove the user themselves

    items_bought_by_user = user_item_matrix.loc[user_id].to_numpy().nonzero()[0]
    n_similar_users = min(10, len(similar_users))  # Consider top 10 or fewer
    similar_user_indices = similar_users.iloc[:n_similar_users].index
    recommended_items = []

    for i in similar_user_indices:
        items_bought_by_similar_user = user_item_matrix.loc[i].to_numpy().nonzero()[0]
        recommended_items.extend(items_bought_by_similar_user)

    recommended_items = [item for item in recommended_items if item not in items_bought_by_user]
    recommended_items = list(set(recommended_items))  # Remove duplicates

    if not recommended_items:
        print(f"No new items to recommend for user {user_id}. Returning popular items.")
        return items_df.sort_values(by='price', ascending=False).head(top_n)

    return items_df[items_df['item_id'].isin(recommended_items)].head(top_n)

# --- Content-Based Filtering ---
def get_item_features(items_df):
    """Creates a TF-IDF representation of item features."""

    items_df['features'] = items_df['category'] + ' ' + items_df['brand'] + ' ' + items_df['color'] + ' ' + items_df['size']
    tfidf_vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf_vectorizer.fit_transform(items_df['features'])
    return tfidf_matrix

def get_item_similarity(tfidf_matrix):
    """Calculates item similarity using cosine similarity."""
    item_similarity = cosine_similarity(tfidf_matrix)
    return item_similarity

def get_content_based_recommendations(item_id, item_similarity, items_df, top_n=10):
    """Generates content-based recommendations."""

    if item_id not in items_df['item_id'].values:
        print(f"Item {item_id} not found. Returning popular items.")
        return items_df.sort_values(by='price', ascending=False).head(top_n)

    item_index = items_df[items_df['item_id'] == item_id].index[0]
    similar_items = pd.Series(item_similarity[item_index], index=items_df.index)
    similar_items = similar_items.sort_values(ascending=False)
    similar_items = similar_items.drop(item_index)  # Remove the item itself

    return items_df.iloc[similar_items.index].head(top_n)

# --- Hybrid Approach (Simple Example) ---
def get_hybrid_recommendations(user_id, item_id, user_item_matrix, user_similarity, tfidf_matrix, items_df, top_n=10):
    """Combines collaborative and content-based recommendations."""

    collaborative_recs = get_user_based_recommendations(user_id, user_item_matrix, user_similarity, items_df, top_n // 2)
    content_recs = get_content_based_recommendations(item_id, get_item_similarity(tfidf_matrix), items_df, top_n // 2)

    if collaborative_recs is None:
      return content_recs
    if content_recs is None:
      return collaborative_recs

    hybrid_recs = pd.concat([collaborative_recs, content_recs]).drop_duplicates()
    return hybrid_recs.head(top_n)

# --- Main Execution ---
user_item = get_user_item_matrix(data)
user_sim = get_user_similarity(user_item)
tfidf = get_item_features(items)

# Example: Get recommendations for a user and a specific item they purchased
example_user_id = data['user_id'].iloc[0]
example_item_id = data['item_id'].iloc[0]

print("\nCollaborative Recommendations for User:", example_user_id)
print(get_user_based_recommendations(example_user_id, user_item, user_sim, items, top_n=5).to_markdown(index=False, numalign="left", stralign="left"))

print("\nContent-Based Recommendations for Item:", example_item_id)
print(get_content_based_recommendations(example_item_id, get_item_similarity(tfidf), items, top_n=5).to_markdown(index=False, numalign="left", stralign="left"))

print("\nHybrid Recommendations for User and Item:", example_user_id, example_item_id)
print(get_hybrid_recommendations(example_user_id, example_item_id, user_item, user_sim, tfidf, items, top_n=5).to_markdown(index=False, numalign="left", stralign="left"))


Collaborative Recommendations for User: 747
| item_id   | category   | brand   | color   | size   | price   | features                 |
|:----------|:-----------|:--------|:--------|:-------|:--------|:-------------------------|
| 3         | Outerwear  | BrandB  | Black   | L      | 181.305 | Outerwear BrandB Black L |
| 16        | Tops       | BrandD  | Blue    | L      | 28.6746 | Tops BrandD Blue L       |
| 21        | Outerwear  | BrandB  | Blue    | L      | 80.2495 | Outerwear BrandB Blue L  |
| 28        | Shoes      | BrandA  | Red     | M      | 100.209 | Shoes BrandA Red M       |
| 30        | Bottoms    | BrandA  | White   | L      | 44.0365 | Bottoms BrandA White L   |

Content-Based Recommendations for Item: 26
| item_id   | category   | brand   | color   | size   | price   | features              |
|:----------|:-----------|:--------|:--------|:-------|:--------|:----------------------|
| 25        | Bottoms    | BrandA  | Blue    | S      | 159.327 | Bottoms BrandA